In [1]:
from pathlib import Path
import math
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from itertools import islice
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import math
from torch.utils.data import WeightedRandomSampler

C:\Users\Xiang Gao\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Loader

In [3]:
def build_chart_manifest(audio_dir, token_dir):
    audio_dir = Path(audio_dir)
    token_dir = Path(token_dir)

    audio_files = {p.stem: p for p in audio_dir.glob("*.npz")}
    token_files = {p.stem: p for p in token_dir.glob("*.json")}

    common_ids = sorted(set(audio_files) & set(token_files))

    rows = []
    for chart_id in common_ids:
        npz_path = audio_files[chart_id]
        json_path = token_files[chart_id]

        audio_arr = np.load(npz_path)["audio_sequences"]
        with open(json_path, "r", encoding="utf-8") as f:
            token_data = json.load(f)

        rows.append({
            "chart_id": chart_id,
            "npz_path": str(npz_path),
            "json_path": str(json_path),
            "n_sequences_audio": int(audio_arr.shape[0]),
            "n_sequences_token": int(len(token_data)),
        })

    manifest_df = pd.DataFrame(rows)
    return manifest_df

def split_chart_manifest(manifest_df, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, random_state=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8

    chart_ids = manifest_df["chart_id"].tolist()

    train_ids, temp_ids = train_test_split(
        chart_ids,
        test_size=(1 - train_ratio),
        random_state=random_state,
        shuffle=True,
    )

    val_size_within_temp = val_ratio / (val_ratio + test_ratio)

    val_ids, test_ids = train_test_split(
        temp_ids,
        train_size=val_size_within_temp,
        random_state=random_state,
        shuffle=True,
    )

    return train_ids, val_ids, test_ids

def build_sequence_index(manifest_df, chart_id_list):
    chart_id_set = set(chart_id_list)

    split_df = manifest_df[manifest_df["chart_id"].isin(chart_id_set)].copy()

    rows = []
    for _, row in split_df.iterrows():
        chart_id = row["chart_id"]
        npz_path = row["npz_path"]
        json_path = row["json_path"]
        n_seq = int(row["n_sequences_audio"])

        for seq_idx in range(n_seq):
            rows.append({
                "chart_id": chart_id,
                "seq_idx": seq_idx,
                "npz_path": npz_path,
                "json_path": json_path,
            })

    seq_index_df = pd.DataFrame(rows)
    return seq_index_df

def load_one_sample(seq_row):
    npz_path = seq_row["npz_path"]
    json_path = seq_row["json_path"]
    seq_idx = int(seq_row["seq_idx"])

    # load audio
    audio_arr = np.load(npz_path)["audio_sequences"]
    audio = audio_arr[seq_idx]

    # load tokens
    with open(json_path, "r", encoding="utf-8") as f:
        token_data = json.load(f)

    item = token_data[seq_idx]
    tokens = item["tokens"]

    return {
        "chart_id": seq_row["chart_id"],
        "seq_idx": seq_idx,
        "audio": audio,
        "tokens": tokens,
        "n_tokens": len(tokens),
    }


def build_vocab_from_all_splits(train_seq_index, val_seq_index, test_seq_index):
    
    token_set = set()

    # 收集所有 json 文件路径
    all_json_paths = set(
        train_seq_index["json_path"].tolist()
        + val_seq_index["json_path"].tolist()
        + test_seq_index["json_path"].tolist()
    )

    for json_path in all_json_paths:
        with open(json_path, "r", encoding="utf-8") as f:
            token_data = json.load(f)

        for item in token_data:
            for tok in item["tokens"]:
                token_set.add(tok)

    special_tokens = ["PAD", "BOS", "EOS", "MASK"]

    # event tokens
    event_tokens = sorted([t for t in token_set if not t.startswith("TS_")])

    # TS tokens 按数字排序
    ts_tokens = sorted(
        [t for t in token_set if t.startswith("TS_")],
        key=lambda x: int(x.split("_")[1])
    )

    vocab_list = special_tokens + event_tokens + ts_tokens

    token_to_id = {tok: i for i, tok in enumerate(vocab_list)}
    id_to_token = {i: tok for tok, i in token_to_id.items()}

    return vocab_list, token_to_id, id_to_token


def encode_tokens(tokens, token_to_id):
    bos = token_to_id["BOS"]
    eos = token_to_id["EOS"]

    token_ids = [token_to_id[t] for t in tokens]

    input_ids = [bos] + token_ids
    labels = token_ids + [eos]

    return input_ids, labels

In [4]:
AUDIO_DIR = r"D:\Study Abroad\course\DSCI498\Project\data\beat_aligned_dataset\audio_npz_diffusion"
TOKEN_DIR = r"D:\Study Abroad\course\DSCI498\Project\data\beat_aligned_dataset\token_json_diffusion"

manifest_df = build_chart_manifest(AUDIO_DIR, TOKEN_DIR)


train_ids, val_ids, test_ids = split_chart_manifest(
    manifest_df,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    random_state=42)

train_seq_index = build_sequence_index(manifest_df, train_ids)
val_seq_index   = build_sequence_index(manifest_df, val_ids)
test_seq_index  = build_sequence_index(manifest_df, test_ids)

vocab_list, token_to_id, id_to_token = build_vocab_from_all_splits(
    train_seq_index,
    val_seq_index,
    test_seq_index
)

In [5]:
class TaikoDataset(Dataset):

    def __init__(self, seq_index_df, token_to_id, min_tokens=1):

        self.token_to_id = token_to_id
        self.samples = []

        print("Filtering empty sequences (one-time scan)...")

        for i in range(len(seq_index_df)):
            row = seq_index_df.iloc[i]
            sample = load_one_sample(row)

            if len(sample["tokens"]) >= min_tokens:
                self.samples.append(row)

            if i % 1000 == 0:
                print(f"Processed {i}/{len(seq_index_df)}")

        print(f"Original size: {len(seq_index_df)}")
        print(f"Filtered size: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        row = self.samples[idx]
        sample = load_one_sample(row)

        input_ids, _ = encode_tokens(sample["tokens"], self.token_to_id)

        return {
            "audio": torch.tensor(sample["audio"], dtype=torch.float32),
            "x0": torch.tensor(input_ids, dtype=torch.long),
        }


def taiko_collate_fn(batch, pad_id=0):

    audio_list = [item["audio"] for item in batch]
    x0_list = [item["x0"] for item in batch]

    # audio: (B, 1536, 128)
    audio = torch.stack(audio_list, dim=0)

    # token padding
    x0 = pad_sequence(x0_list, batch_first=True, padding_value=pad_id)

    attention_mask = (x0 != pad_id).long()

    return {
        "audio": audio,
        "x0": x0,
        "attention_mask": attention_mask,
    }

In [6]:
BATCH_SIZE = 32

train_dataset = TaikoDataset(train_seq_index, token_to_id)
val_dataset   = TaikoDataset(val_seq_index, token_to_id)
test_dataset  = TaikoDataset(test_seq_index, token_to_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda batch: taiko_collate_fn(batch, pad_id=token_to_id["PAD"])
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: taiko_collate_fn(batch, pad_id=token_to_id["PAD"])
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda batch: taiko_collate_fn(batch, pad_id=token_to_id["PAD"])
)

Filtering empty sequences (one-time scan)...
Processed 0/3025
Processed 1000/3025
Processed 2000/3025
Processed 3000/3025
Original size: 3025
Filtered size: 2984
Filtering empty sequences (one-time scan)...
Processed 0/393
Original size: 393
Filtered size: 393
Filtering empty sequences (one-time scan)...
Processed 0/354
Original size: 354
Filtered size: 351


# Transformer architecture

In [7]:
class AudioEmbedding(nn.Module):
    def __init__(self, input_dim=128, d_model=256):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)

    def forward(self, x):
        # x: (B, 1536, 128)
        return self.proj(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model=256, max_len=2048):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]


class AudioEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4, dim_feedforward=1024, dropout=0.1):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        return self.encoder(x)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

    def forward(self, input_ids):
        return self.embed(input_ids)


class ChartDecoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4, dim_feedforward=1024, dropout=0.1):
        super().__init__()

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        return self.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )


class OutputHead(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(x)

In [8]:
class TaikoTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        input_dim=128,
        d_model=256,
        nhead=4,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.3,
        max_len=2048,
    ):
        super().__init__()

        # ===== encoder =====
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.randn(1, max_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        # ===== decoder =====
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_decoder = nn.Parameter(torch.randn(1, max_len, d_model))

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )

        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, audio, input_ids, decoder_attention_mask=None):
        # ===== encoder =====
        audio = self.input_proj(audio)
        audio = audio + self.pos_encoder[:, :audio.size(1), :]
        memory = self.encoder(audio)

        # ===== decoder input =====
        tok_x = self.token_embed(input_ids)
        tok_x = tok_x + self.pos_decoder[:, :tok_x.size(1), :]

        # ===== decoder =====
        out = self.decoder(
            tgt=tok_x,
            memory=memory,
            tgt_key_padding_mask=(decoder_attention_mask == 0)
        )

        logits = self.output_layer(out)
        return logits

# Training

In [9]:
def compute_complexity(x):
    tokens = [id_to_token[int(t)] for t in x]

    ts_values = []
    note_seq = []

    for t in tokens:
        if t.startswith("TS_"):
            val = int(t.split("_")[1])
            ts_values.append(val)
        else:
            note_seq.append(t)

    score = 0

    # ===== 1️⃣ TS变化（但限制范围）
    valid_ts = [v for v in ts_values if v <= 96]  # 关键限制

    if len(valid_ts) > 1:
        score += np.std(valid_ts) * 0.05

    # ===== 2️⃣ TS多样性（合理范围内）
    score += len(set(valid_ts)) * 1.0

    # ===== 3️⃣ 惩罚异常TS（关键！）
    for v in ts_values:
        if v > 120:
            score -= 2.0  # 惩罚极端值

    # ===== 4️⃣ pattern多样性
    note_counter = Counter(note_seq)
    score += len(note_counter) * 1.5

    # ===== 5️⃣ 连续音符（burst）
    burst = 0
    max_burst = 0

    for t in tokens:
        if not t.startswith("TS_"):
            burst += 1
            max_burst = max(max_burst, burst)
        else:
            burst = 0

    score += max_burst * 1.0

    # ===== 6️⃣ 特殊token（小权重）
    for t in tokens:
        if "BIG" in t:
            score += 0.5
        elif t == "DRUMROLL":
            score += 1.0

    return score

In [ ]:
def collate_fn(batch):
    audio_list = [b["audio"] for b in batch]
    x0_list = [b["x0"] for b in batch]

    # 找最大长度
    max_len = max(x.shape[0] for x in x0_list)

    padded_x0 = []
    attention_mask = []

    for x in x0_list:
        pad_len = max_len - x.shape[0]

        padded = torch.cat([
            x,
            torch.full((pad_len,), PAD_ID)
        ])

        mask = torch.cat([
            torch.ones(x.shape[0]),
            torch.zeros(pad_len)
        ])

        padded_x0.append(padded)
        attention_mask.append(mask)

    padded_x0 = torch.stack(padded_x0)
    attention_mask = torch.stack(attention_mask)

    audio = torch.stack(audio_list)

    return {
        "audio": audio,
        "x0": padded_x0.long(),
        "attention_mask": attention_mask.long()
    }

In [18]:
scores = [compute_complexity(train_dataset[i]["x0"]) for i in range(len(train_dataset))]

scores = np.array(scores)

# 归一化（很重要）
scores = (scores - scores.min()) / (scores.max() - scores.min())

# 控制强度（关键参数）
alpha = 2.0

sample_weights = 1.0 + alpha * scores

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,  # ✔ 用 sampler
    collate_fn=lambda batch: taiko_collate_fn(batch, pad_id=token_to_id["PAD"])  # ✔ 保留
)

In [19]:
def corrupt_tokens(x0, attention_mask, mask_id):
    B, L = x0.shape
    device = x0.device

    x_t = x0.clone()
    corrupt_pos = torch.zeros_like(x0, dtype=torch.bool)

    valid = (attention_mask == 1) & (x0 != PAD_ID)

    for b in range(B):
        valid_idx = torch.where(valid[b])[0]

        if len(valid_idx) < 4:
            continue

        # ===== 1️⃣ 轻度随机 mask（保留）=====
        num_mask = int(0.10 * len(valid_idx))
        if num_mask > 0:
            mask_idx = valid_idx[torch.randperm(len(valid_idx))[:num_mask]]
            x_t[b, mask_idx] = mask_id
            corrupt_pos[b, mask_idx] = True

        # ===== 2️⃣ DON/KAT flip（核心）=====
        for i in valid_idx:
            token = x_t[b, i].item()

            if token == token_to_id["DON"] and torch.rand(1) < 0.25:
                x_t[b, i] = token_to_id["KAT"]
                corrupt_pos[b, i] = True

            elif token == token_to_id["KAT"] and torch.rand(1) < 0.25:
                x_t[b, i] = token_to_id["DON"]
                corrupt_pos[b, i] = True

        # ===== 3️⃣ 局部重复打破（关键）=====
        for i in range(len(valid_idx) - 2):
            idx = valid_idx[i:i+3]

            tokens = x_t[b, idx]

            # 连续相同（DON DON DON / KAT KAT KAT）
            if (tokens == tokens[0]).all():
                if torch.rand(1) < 0.7:
                    for j in idx:
                        if torch.rand(1) < 0.5:
                            x_t[b, j] = token_to_id["KAT"] if tokens[0] == token_to_id["DON"] else token_to_id["DON"]
                            corrupt_pos[b, j] = True

        # ===== 4️⃣ 局部 block 扰动（不动 TS）=====
        if len(valid_idx) > 6:
            start = torch.randint(0, len(valid_idx) - 4, (1,)).item()
            block = valid_idx[start:start+4]

            for i in block:
                token = x_t[b, i].item()

                # 不动 TS
                token_str = None
                for k, v in token_to_id.items():
                    if v == token:
                        token_str = k
                        break

                if token_str and not token_str.startswith("TS_"):
                    if torch.rand(1) < 0.5:
                        x_t[b, i] = mask_id
                        corrupt_pos[b, i] = True

    return x_t, corrupt_pos


def diffusion_loss(logits, x0, mask_pos):
    logits_masked = logits[mask_pos]
    targets = x0[mask_pos]

    if logits_masked.numel() == 0:
        return logits.sum() * 0.0

    return F.cross_entropy(logits_masked, targets)

In [20]:
def smart_block_mask(x0, attention_mask, mask_id, block_size=8):
    B, L = x0.shape
    device = x0.device

    x_t = x0.clone()
    mask_pos = torch.zeros_like(x0, dtype=torch.bool)

    id_to_token = {v: k for k, v in token_to_id.items()}

    for b in range(B):

        valid_len = attention_mask[b].sum().item()

        if valid_len <= block_size + 2:
            continue

        # ===== 1️⃣ 找“interesting位置” =====
        interesting = []

        for i in range(1, valid_len - 1):
            token = x0[b, i].item()
            token_str = id_to_token[token]

            # BIG token
            if "BIG" in token_str:
                interesting.append(i)

            # TS变化（简单检测）
            if token_str.startswith("TS_"):
                if i > 0:
                    prev = id_to_token[x0[b, i-1].item()]
                    if prev.startswith("TS_") and prev != token_str:
                        interesting.append(i)

        # ===== 2️⃣ 如果找不到，就 fallback =====
        if len(interesting) == 0:
            start = torch.randint(1, valid_len - block_size, (1,)).item()
        else:
            center = interesting[torch.randint(len(interesting), (1,)).item()]
            start = max(1, center - block_size // 2)
            start = min(start, valid_len - block_size)

        end = start + block_size

        # ===== 3️⃣ mask =====
        x_t[b, start:end] = mask_id
        mask_pos[b, start:end] = True

    return x_t, mask_pos

In [21]:
# ====== Diffusion Step 1: Mask (discrete forward) ======
# 1️⃣ 正确的 MASK / VOCAB（不再手动 +1）
MASK_ID = token_to_id["MASK"]
PAD_ID = token_to_id["PAD"]
VOCAB_SIZE = len(token_to_id)


# 2️⃣ 构造模型
model = TaikoTransformer(
    vocab_size=VOCAB_SIZE,
    input_dim=128,
    d_model=256,
    nhead=4,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.3,
    max_len=2048,  
)

model = model.to(device)

In [22]:
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

MASK_ID = token_to_id["MASK"]
PAD_ID = token_to_id["PAD"]

num_epochs = 5
print_every = 50

# 👉 推荐稳定范围（你之前验证过是好的）
MIN_MASK_RATIO = 0.10
MAX_MASK_RATIO = 0.30

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1} ===")

    # =========================
    # 🔵 TRAIN
    # =========================
    model.train()
    train_loss = 0.0
    train_steps = 0

    for step, batch in enumerate(train_loader):
        audio = batch["audio"].to(device)
        x0 = batch["x0"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        B = x0.size(0)

        # 👉 per-sample 随机 mask
        mask_ratio = torch.rand(B, device=device) * (MAX_MASK_RATIO - MIN_MASK_RATIO) + MIN_MASK_RATIO
        x_t, mask_pos = smart_block_mask(x0,attention_mask,MASK_ID,block_size=8)

        logits = model(
            audio,
            x_t,
            decoder_attention_mask=attention_mask
        )

        # 👉 只在被 mask 的位置算 loss（核心）
        loss = diffusion_loss(logits, x0, mask_pos)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_steps += 1

        if step % print_every == 0:
            print(
                f"[train] step {step} | loss {loss.item():.4f} | "
                f"mask_ratio {mask_ratio.mean().item():.4f}"
            )

    print(f"[train] epoch avg loss: {train_loss / train_steps:.4f}")

    # =========================
    # 🟡 VALIDATION
    # =========================
    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in val_loader:
            audio = batch["audio"].to(device)
            x0 = batch["x0"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            B = x0.size(0)

            # 👉 固定验证 mask（方便比较）
            mask_ratio = torch.full((B,), 0.20, device=device)
            x_t, mask_pos = smart_block_mask(x0,attention_mask,MASK_ID,block_size=8)

            logits = model(
                audio,
                x_t,
                decoder_attention_mask=attention_mask
            )

            loss = diffusion_loss(logits, x0, mask_pos)

            val_loss += loss.item()
            val_steps += 1

    print(f"[val] epoch avg loss: {val_loss / val_steps:.4f}")


=== Epoch 1 ===


C:\Users\Xiang Gao\AppData\Roaming\Python\Python311\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


[train] step 0 | loss 5.2068 | mask_ratio 0.2080
[train] step 50 | loss 2.4404 | mask_ratio 0.2187
[train] epoch avg loss: 2.4439
[val] epoch avg loss: 2.0905

=== Epoch 2 ===
[train] step 0 | loss 2.2570 | mask_ratio 0.2089
[train] step 50 | loss 1.8155 | mask_ratio 0.2058
[train] epoch avg loss: 1.9234
[val] epoch avg loss: 1.6439

=== Epoch 3 ===
[train] step 0 | loss 1.7130 | mask_ratio 0.1908
[train] step 50 | loss 1.6460 | mask_ratio 0.1999
[train] epoch avg loss: 1.5285
[val] epoch avg loss: 1.4028

=== Epoch 4 ===
[train] step 0 | loss 1.4826 | mask_ratio 0.1991
[train] step 50 | loss 1.5500 | mask_ratio 0.2205
[train] epoch avg loss: 1.4616
[val] epoch avg loss: 1.4040

=== Epoch 5 ===
[train] step 0 | loss 1.4507 | mask_ratio 0.2037
[train] step 50 | loss 1.3345 | mask_ratio 0.1886
[train] epoch avg loss: 1.4123
[val] epoch avg loss: 1.3775


In [36]:
model.eval()

sample_idx = 5  # 可以多试几个

batch = val_loader.dataset[sample_idx]

audio = batch["audio"].unsqueeze(0).to(device)
x0 = batch["x0"].unsqueeze(0).to(device)

attention_mask = (x0 != PAD_ID).long().to(device)

# ===== mask =====
x_t, mask_pos = smart_block_mask(
    x0,
    attention_mask,
    MASK_ID,
    block_size=8
)

# ===== forward =====
with torch.no_grad():
    logits = model(audio, x_t, decoder_attention_mask=attention_mask)

# ===== 👉 纯 sampling（关键）=====
temperature = 1.2

probs = torch.softmax(logits / temperature, dim=-1)

B, L, V = probs.shape
sampled = torch.multinomial(probs.view(-1, V), num_samples=1).view(B, L)

# ===== fill back =====
pred = x_t.clone()
pred[mask_pos] = sampled[mask_pos]

# ===== decode =====
id_to_token = {v: k for k, v in token_to_id.items()}

def decode(seq):
    return [id_to_token[int(x)] for x in seq]

# ===== 只看mask附近 =====
mask_indices = torch.where(mask_pos[0])[0]

if len(mask_indices) == 0:
    print("No masked positions found.")
else:
    start = max(0, mask_indices[0].item() - 1)
    end = min(x0.shape[1], mask_indices[-1].item() + 2)

    print(f"Showing window: [{start}:{end}]")

    print("\n=== ORIGINAL (local) ===")
    print(decode(x0[0][start:end]))

    print("\n=== MASKED (local) ===")
    print(decode(x_t[0][start:end]))

    print("\n=== PREDICTION (pure sampling) ===")
    print(decode(pred[0][start:end]))

Showing window: [28:38]

=== ORIGINAL (local) ===
['TS_12', 'KAT', 'TS_24', 'DON', 'TS_24', 'BIGDON', 'TS_48', 'DON', 'TS_24', 'KAT']

=== MASKED (local) ===
['TS_12', 'MASK', 'MASK', 'MASK', 'MASK', 'MASK', 'MASK', 'MASK', 'MASK', 'KAT']

=== PREDICTION (pure sampling) ===
['TS_12', 'DON', 'TS_12', 'BIGKAT', 'TS_24', 'KAT', 'TS_12', 'BIGDON', 'TS_48', 'KAT']


# Refine AR Output

In [47]:
import torch
import numpy as np
import librosa

# =========================
# 🔧 基本参数（必须和训练一致）
# =========================
SR = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 30
FMAX = 8000

FRAMES_PER_BEAT = 48
BLOCK_BEATS = 32
FRAMES_PER_BLOCK = FRAMES_PER_BEAT * BLOCK_BEATS  # 1536

# =========================
# 🔧 NOTE 判断函数（核心）
# =========================
def is_note(token):
    return not token.startswith("TS_") and token not in ["PAD", "BOS", "EOS"]

# =========================
# 🔧 只 mask NOTE
# =========================
def mask_notes_only(x0, token_list, mask_ratio, mask_id):
    x_t = x0.clone()
    mask_pos = torch.zeros_like(x0, dtype=torch.bool)

    for i, t in enumerate(token_list):
        if is_note(t):
            if np.random.rand() < mask_ratio:
                x_t[0, i] = mask_id
                mask_pos[0, i] = True

    return x_t, mask_pos

# =========================
# 🔧 audio → aligned mel
# =========================
def build_aligned_mel(audio_path, offset_ms, bpm):
    y, _ = librosa.load(audio_path, sr=SR, mono=True)

    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max).T

    beat_ms = bpm
    frame_ms = beat_ms / FRAMES_PER_BEAT

    duration_ms = len(y) / SR * 1000
    total_frames = int((duration_ms - offset_ms) // frame_ms)

    raw_times = np.arange(mel_db.shape[0]) * (HOP_LENGTH / SR * 1000)
    target_times = offset_ms + np.arange(total_frames) * frame_ms

    aligned = np.zeros((total_frames, N_MELS), dtype=np.float32)

    for m in range(N_MELS):
        aligned[:, m] = np.interp(
            target_times, raw_times, mel_db[:, m]
        )

    return aligned

# =========================
# 🔧 数据准备（你已有）
# =========================
block_idx = 0
block = blocks[block_idx]

id_to_token = {v: k for k, v in token_to_id.items()}

def encode(block):
    return [token_to_id[t] for t in block]

def decode(seq):
    return [id_to_token[int(x)] for x in seq]

# =========================
# 🔧 token → tensor
# =========================
x0 = torch.tensor(encode(block), dtype=torch.long).unsqueeze(0).to(device)
attention_mask = torch.ones_like(x0).to(device)

# =========================
# 🔧 audio 对齐（关键）
# =========================
audio_path = r"D:\Study Abroad\course\DSCI498\Project\data\unpacked\34699\Hoshiful 2nd op.mp3"
offset_ms = 232
bpm = 60000 / 342.857142857143

aligned = build_aligned_mel(audio_path, offset_ms, bpm)

start = block_idx * FRAMES_PER_BLOCK
end = start + FRAMES_PER_BLOCK

audio_block = aligned[start:end]

audio = torch.tensor(audio_block, dtype=torch.float32).unsqueeze(0).to(device)

# =========================
# 🔥 mask（只动 NOTE）
# =========================
MASK_ID = token_to_id["MASK"]

x_t, mask_pos = mask_notes_only(
    x0,
    block,
    mask_ratio=0.4,
    mask_id=MASK_ID
)

# =========================
# 🔥 diffusion forward
# =========================
model.eval()
with torch.no_grad():
    logits = model(audio, x_t, decoder_attention_mask=attention_mask)

# =========================
# 🔥 sampling
# =========================
temperature = 1.1
probs = torch.softmax(logits / temperature, dim=-1)

B, L, V = probs.shape
sampled = torch.multinomial(probs.view(-1, V), 1).view(B, L)

# =========================
# 🔥 fill back
# =========================
pred = x_t.clone()
pred[mask_pos] = sampled[mask_pos]

# =========================
# 🔥 输出对比
# =========================
orig_tokens = decode(x0[0])
masked_tokens = decode(x_t[0])
pred_tokens = decode(pred[0])

mask_indices = torch.where(mask_pos[0])[0]

if len(mask_indices) > 0:
    start = max(0, mask_indices[0].item() - 3)
    end = min(len(orig_tokens), mask_indices[-1].item() + 4)

    print(f"\nShowing window: [{start}:{end}]\n")

    print(f"{'idx':<4} | {'ORIGINAL':<12} | {'MASKED':<12} | {'REPAIRED':<12}")
    print("-" * 55)

    for i in range(start, end):
        o = orig_tokens[i]
        m = masked_tokens[i]
        p = pred_tokens[i]

        # 标记被 mask 的位置
        mark = "*" if i in mask_indices.tolist() else " "

        print(f"{i:<4} | {o:<12} | {m:<12} | {p:<12} {mark}")

else:
    print("No mask applied.")


Showing window: [0:109]

idx  | ORIGINAL     | MASKED       | REPAIRED    
-------------------------------------------------------
0    | DON          | DON          | DON           
1    | TS_48        | TS_48        | TS_48         
2    | KAT          | MASK         | DON          *
3    | TS_48        | TS_48        | TS_48         
4    | KAT          | MASK         | BIGKAT       *
5    | TS_48        | TS_48        | TS_48         
6    | KAT          | MASK         | DON          *
7    | TS_48        | TS_48        | TS_48         
8    | DON          | MASK         | KAT          *
9    | TS_48        | TS_48        | TS_48         
10   | KAT          | MASK         | KAT          *
11   | TS_24        | TS_24        | TS_24         
12   | DON          | MASK         | DON          *
13   | TS_24        | TS_24        | TS_24         
14   | KAT          | KAT          | KAT           
15   | TS_24        | TS_24        | TS_24         
16   | DON          | DON          |